In [5]:
import mlflow
from mlflow import MlflowClient
client = MlflowClient(registry_uri="sqlite:///C:/ESG/mlflow.db")
models = client.get_registered_model(name="my-esg-catboost-model")
print(models)

<RegisteredModel: aliases={'staging': 3}, creation_timestamp=1763554765216, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1764063057210, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1764063057210, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1764063057210, metrics=None, model_id=None, name='my-esg-catboost-model', params=None, run_id='576f88e8894444658d676423d31e6029', run_link=None, source='runs:/576f88e8894444658d676423d31e6029/model', status='READY', status_message=None, tags={'metric_R2': '0.9998938875643146',
 'metric_RMSE': '0.32968592280825726',
 'param_bagging_temperature': '0.07045816140691463',
 'param_border_count': '227',
 'param_depth': '7',
 'param_iterations': '999',
 'param_l2_leaf_reg': '2.1557816466134136',
 'param_learning_rate': '0.18941284157334615',
 'param_loss_function': 'RMSE',
 'param_model_path': 'C:\\ESG\\models\\catboost_final_model.pkl',
 'param_ran

In [6]:
import mlflow
from mlflow import MlflowClient
client = MlflowClient(registry_uri="sqlite:///C:/ESG/mlflow.db")
versions = client.search_model_versions('name="my-esg-catboost-model"')
print([v.version for v in versions])

[3, 2, 1]


In [7]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import json

# -----------------------------
# 1. Load your CatBoost model
# -----------------------------
model_path = "C:\ESG\mlruns\\0\\576f88e8894444658d676423d31e6029\\artifacts\\model\\catboost_model.cbm"   # <-- change if needed
model = CatBoostRegressor()
model.load_model(model_path)

print("📌 Loaded model!")
print("MODEL FEATURES:", model.feature_names_)
print("=" * 120)

# -----------------------------
# 2. Create sample input (same as API payload)
# -----------------------------
sample_json = {
    "CompanyID": 101,
    "Year": 2024,
    "Revenue": 987654321,
    "ProfitMargin": 14.8,
    "MarketCap": 5500000000,
    "CarbonEmissions": 1200,
    "WaterUsage": 45000,
    "EnergyConsumption": 78000,
    "ESG_Environmental": 72,
    "ESG_Social": 68,
    "ESG_Governance": 75,
    "ESG_Overall": 71,
    "ESG_OVERALL_PREV": 69
}

print("Input JSON:")
print(sample_json)
print("=" * 120)

# -----------------------------
# 3. Convert JSON → DataFrame
# -----------------------------
df = pd.DataFrame([sample_json])
print("Df BEFORE feature engineering:", df.columns.tolist())
print("=" * 120)

# -----------------------------
# 4. Apply your feature engineering
# -----------------------------
from utils import feature_engineering_for_inference   # <-- ensure utils.py is in same folder

df_fe = feature_engineering_for_inference(df)

print("Df AFTER feature engineering:", df_fe.columns.tolist())
print("=" * 120)

# -----------------------------
# 5. Compare model features vs df features
# -----------------------------
model_feats = model.feature_names_
df_feats = df_fe.columns.tolist()

missing_in_df = [f for f in model_feats if f not in df_feats]
extra_in_df   = [f for f in df_feats if f not in model_feats]

print("❌ Missing in DF:", missing_in_df)
print("⚠️ Extra in DF:", extra_in_df)

print("=" * 120)

# -----------------------------
# 6. Optional: Try prediction (only if ALL features match)
# -----------------------------
if len(missing_in_df) == 0:
    pred = model.predict(df_fe)
    print("Prediction:", pred[0])
else:
    print("❌ Cannot predict because features do not match.")


📌 Loaded model!
MODEL FEATURES: ['CompanyID', 'CompanyName', 'Industry', 'Region', 'Year', 'Revenue', 'ProfitMargin', 'MarketCap', 'GrowthRate', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance', 'CarbonEmissions', 'WaterUsage', 'EnergyConsumption', 'missing_count', 'Revenue_per_carbon', 'Revenue_per_Water', 'Revenue_per_Energy', 'ProfitMargin_x_ESG', 'ESG_Pillar_Mean', 'ESG_Pillar_Std', 'Year_Since_2015', 'Year_Normalised', 'ESG_Overall_Lag1', 'ESG_Overall_RollingMean', 'ESG_Overall_RollingStd']
Input JSON:
{'CompanyID': 101, 'Year': 2024, 'Revenue': 987654321, 'ProfitMargin': 14.8, 'MarketCap': 5500000000, 'CarbonEmissions': 1200, 'WaterUsage': 45000, 'EnergyConsumption': 78000, 'ESG_Environmental': 72, 'ESG_Social': 68, 'ESG_Governance': 75, 'ESG_Overall': 71, 'ESG_OVERALL_PREV': 69}
Df BEFORE feature engineering: ['CompanyID', 'Year', 'Revenue', 'ProfitMargin', 'MarketCap', 'CarbonEmissions', 'WaterUsage', 'EnergyConsumption', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance', 